# Customer Churn Prediction — Modeling Walkthrough

This notebook runs the same code as `train.py`, step by step, so each stage can be inspected:

1. Load data from SQLite and do a stratified 80/20 train/test split
2. Build one `imblearn` pipeline: **feature engineering → one-hot encoding → SMOTE → model-based feature selection → classifier**
3. Tune **Random Forest, XGBoost and LightGBM** with `RandomizedSearchCV` (stratified 5-fold CV, refit on ROC-AUC)
4. Pick the best model by **CV ROC-AUC**, then evaluate all models once on the test set with **ROC-AUC, Precision, Recall and F1**

Run `python ingestion_db.py` first.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, ConfusionMatrixDisplay

from features import load_and_split_data, FeatureEngineer, build_preprocessor
from train import build_pipeline, setup_models, train_all, RANDOM_STATE

## 1. Load and split

In [2]:
X_train, X_test, y_train, y_test = load_and_split_data(random_state=RANDOM_STATE)
print("Train churn rate:", round(y_train.mean(), 3), "| Test churn rate:", round(y_test.mean(), 3))

Rows: 7043 | Churn rate: 26.5% | Train: 5634 | Test: 1409
Train churn rate: 0.265 | Test churn rate: 0.265


The split is **stratified**, so train and test keep the same 26.5% churn rate. It happens *before* any step that learns from data, so the test set never influences training.

## 2. Feature engineering and encoding

In [3]:
fe = FeatureEngineer().fit(X_train)
engineered = fe.transform(X_train)
new_cols = ["tenure_segment", "is_month_to_month", "has_long_contract", "service_bundle_score",
            "charges_per_tenure", "high_monthly_charges", "high_risk_payment", "single_customer"]
print("Median MonthlyCharges learned on train only:", fe.median_monthly_)
engineered[new_cols].head()

Median MonthlyCharges learned on train only: 70.5


,tenure_segment,is_month_to_month,has_long_contract,service_bundle_score,charges_per_tenure,high_monthly_charges,high_risk_payment,single_customer
3738,2-4yr,1,0,1,47.268056,0,1,1
3151,1-2yr,1,0,1,71.971875,1,0,0
4860,1-2yr,0,1,3,42.167857,0,0,0
3867,2-4yr,0,1,2,70.581481,1,0,0
3810,0-6m,1,0,0,22.275000,0,1,0


In [4]:
pre = build_preprocessor(X_train).fit(engineered)
encoded = pre.transform(engineered)
print("Encoded feature matrix:", encoded.shape)

Encoded feature matrix: (5634, 53)


## 3. What SMOTE does

In [5]:
from imblearn.over_sampling import SMOTE
_, y_res = SMOTE(random_state=RANDOM_STATE).fit_resample(encoded, y_train)
pd.DataFrame({"before SMOTE": y_train.value_counts(), "after SMOTE": pd.Series(y_res).value_counts()})

,before SMOTE,after SMOTE
Churn,,
0,4139,4139
1,1495,4139


SMOTE creates synthetic churners by interpolating between a churner and its nearest churner neighbours, until both classes are equal.

**Important:** in the real pipeline SMOTE is a step *inside* the `imblearn` Pipeline. During cross-validation it is applied only to the training folds, never to the validation fold or the test set. If SMOTE were applied before CV, synthetic points derived from validation rows would leak into training and inflate the CV scores.

## 4. The full pipeline

In [6]:
models = setup_models()
pipe = build_pipeline(models["XGBoost"][0], X_train)
pipe

,steps,"[('features', ...), ('preprocess', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feature_name}__{transformer_name}""``. See :meth:`str.format` method from the standard library for more info... versionadded:: 1.0.. versionchanged:: 1.6 `verbose_feature_names_out` can be a callable or a string to be formatted.",False
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transforme

`SelectFromModel` fits a Random Forest on the (SMOTE-balanced) training fold and keeps only features whose importance is above a threshold. The threshold itself (`mean`, `median` or `0.5*mean`) is tuned as a hyperparameter.

## 5. Tune and compare Random Forest, XGBoost and LightGBM

In [7]:
results = train_all(X_train, y_train, X_test, y_test)


Tuning RandomForest (10 combinations x 5 folds)...


CV   ROC-AUC 0.8418 | Recall 0.6428 | Precision 0.6018 | F1 0.6212
TEST ROC-AUC 0.8383 | Recall 0.6257 | Precision 0.5666 | F1 0.5947 | Accuracy 0.7736
Features kept by SelectFromModel: 28

Tuning XGBoost (10 combinations x 5 folds)...


CV   ROC-AUC 0.8433 | Recall 0.5846 | Precision 0.6272 | F1 0.6047
TEST ROC-AUC 0.8458 | Recall 0.5775 | Precision 0.6085 | F1 0.5926 | Accuracy 0.7892
Features kept by SelectFromModel: 28

Tuning LightGBM (10 combinations x 5 folds)...


CV   ROC-AUC 0.8380 | Recall 0.6214 | Precision 0.6163 | F1 0.6187
TEST ROC-AUC 0.8345 | Recall 0.6016 | Precision 0.5921 | F1 0.5968 | Accuracy 0.7842
Features kept by SelectFromModel: 14


In [8]:
rows = []
for name, r in results.items():
    t = r["test"]
    rows.append({"Model": name, "CV ROC-AUC": r["cv"]["roc_auc"], "Test ROC-AUC": t["roc_auc"],
                 "Precision": t["precision"], "Recall": t["recall"], "F1": t["f1"],
                 "Accuracy": t["accuracy"], "Features kept": len(r["features"])})
table = pd.DataFrame(rows).set_index("Model").round(4)
best_name = table["CV ROC-AUC"].idxmax()
print("Best model by CV ROC-AUC:", best_name)
table

Best model by CV ROC-AUC: XGBoost


,CV ROC-AUC,Test ROC-AUC,Precision,Recall,F1,Accuracy,Features kept
Model,,,,,,,
RandomForest,0.8418,0.8383,0.5666,0.6257,0.5947,0.7736,28
XGBoost,0.8433,0.8458,0.6085,0.5775,0.5926,0.7892,28
LightGBM,0.8380,0.8345,0.5921,0.6016,0.5968,0.7842,14


- The best model is chosen by **CV ROC-AUC**, not by test score, so the test set stays an unbiased final check.
- CV and test ROC-AUC are close for every model, which indicates good generalisation and no leakage.
- Precision / Recall / F1 are for the churn class at the default 0.5 threshold.

## 6. ROC curves

In [9]:
plt.figure(figsize=(7, 6))
for name, r in results.items():
    fpr, tpr, _ = roc_curve(y_test, r["pipeline"].predict_proba(X_test)[:, 1])
    plt.plot(fpr, tpr, label=f"{name} (AUC = {r['test']['roc_auc']:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Random (AUC = 0.500)")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate (Recall)")
plt.title("ROC Curves - Test Set"); plt.legend(); plt.show()

## 7. Confusion matrices

In [10]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, r) in zip(axes, results.items()):
    ConfusionMatrixDisplay(np.array(r["test"]["confusion_matrix"]),
                           display_labels=["Stay", "Churn"]).plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(name)
plt.tight_layout(); plt.show()

## 8. Model-based feature selection and importance

In [11]:
for name, r in results.items():
    print(f"{name}: kept {len(r['features'])} of {encoded.shape[1]} features")

RandomForest: kept 28 of 53 features
XGBoost: kept 28 of 53 features
LightGBM: kept 14 of 53 features


In [12]:
best = results[best_name]["pipeline"]
imp = pd.Series(best.named_steps["model"].feature_importances_,
                index=results[best_name]["features"]).sort_values()
imp.plot(kind="barh", figsize=(8, 9), title=f"Feature importance - {best_name}")
plt.tight_layout(); plt.show()

Contract type, tenure, payment method, internet service and add-on support services dominate — the same drivers found in `eda.ipynb`.

## 9. Business takeaway
- **XGBoost** ranks customers by churn risk best (highest ROC-AUC) → use its probability to prioritise retention calls.
- **Random Forest** has the highest recall → better if missing a churner is much more costly than an unnecessary offer.
- The 0.5 threshold can be moved along the precision-recall curve to match the retention budget.